<a href="https://colab.research.google.com/github/franklinnixon1102-a11y/devops/blob/main/agentAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install gtts

In [17]:
import os, json, gradio as gr
!pip install gtts
from gtts import gTTS
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

def run(code, steps, idx, d):
  if code is not None:
    try:
          res = client.models.generate_content(
              model="gemini-3.6-flash",
              contents=f"Break down into JSON array of {{code, explanation}}:\n{code}",
              config=types.GenerateContentConfig(response_mime_type="application/json")
          )
          steps = json.loads(res.text)
          for i, s in enumerate(steps):
            gTTS(f"Line {i+1}: {s['code']}. {s['explanation']}").save(f"s_{i}.mp3")
            s['audio'] = f"s_{i}.mp3"
          idx = 0
    except Exception as e: return [], 0, f"Error: {e}", None, "0/0"

  if not steps: return [], 0, "No code.", None, "0/0"
  idx = max(0, min(len(steps) - 1, idx + d))
  s = steps[idx]
  return steps, idx, f"python\n{s['code']}\n\n{s['explanation']}", s['audio'], f"{idx+1}/{len(steps)}"

with gr.Blocks(title="Code Explainer") as demo:
  steps, idx = gr.State([]), gr.State(0)
  with gr.Row():
    with gr.Column():
      inp = gr.Code(value='', language="python")
      btn = gr.Button("Analyze", variant="primary")
      key = gr.Textbox(placeholder="Press Enter to step...")
    with gr.Column():
      prog = gr.Markdown("0/0")
      out = gr.Markdown()
      aud = gr.Audio(autoplay=True)
      with gr.Row():
        prev = gr.Button("<")
        nxt = gr.Button("->", variant="primary")


  out_list = [steps, idx, out, aud, prog]
  btn.click(lambda c, s, i: run(c, s, i, 0), [inp, steps, idx], out_list)
  nxt.click(lambda s, i: run(None, s, i, 1), [steps, idx], out_list)
  key.submit(lambda s, i: run(None, s, i, 1), [steps, idx], out_list)
  prev.click(lambda s, i: run(None, s, i, -1), [steps, idx], out_list)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b153044394a0210142.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
